<a href="https://colab.research.google.com/github/maliksonmez/arnavutkoy-air-quality-forecast/blob/main/Arnavutkoy_Gunluk_Tahmin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

kirleticiler = ["PM10", "PM2_5", "SO2", "NO2"]

print("Tahminleme araçları hazır.")

Tahminleme araçları hazır.


In [ ]:
yuklenen = files.upload()
dosya_adi = next(iter(yuklenen))

Saving İstanbul - Arnavutköy_16.09.2023-16.09.2026_Hava Kalitesi - İstasyon Veri İndirme_Detay.xlsx to İstanbul - Arnavutköy_16.09.2023-16.09.2026_Hava Kalitesi - İstasyon Veri İndirme_Detay.xlsx


In [ ]:
ham_veri = pd.read_excel(dosya_adi, header=None)

veri = ham_veri.iloc[2:].copy()
veri.columns = ["tarih"] + kirleticiler
veri = veri.reset_index(drop=True)

veri["tarih"] = pd.to_datetime(veri["tarih"], errors="coerce")

for k in kirleticiler:
    metin = veri[k].astype("string").str.strip()
    veri[k] = pd.to_numeric(
        metin.str.replace(",", ".", regex=False),
        errors="coerce"
    ).astype(float)

veri = veri.sort_values("tarih").reset_index(drop=True)

# Tarih sorunları varsa ilerlemeden kontrol edelim.
assert veri["tarih"].notna().all(), "Geçersiz tarih var."
assert not veri["tarih"].duplicated().any(), "Tekrarlı tarih var."
assert veri["tarih"].diff().dropna().eq(
    pd.Timedelta(hours=1)
).all(), "Saatlik zaman çizelgesinde düzensizlik var."

# Ham veriyi koruyarak analiz kopyasını oluştur.
eda_veri = veri.copy()

for k in ["PM10", "PM2_5"]:
    eda_veri.loc[eda_veri[k].eq(985), k] = np.nan

saatlik = eda_veri.set_index("tarih")[kirleticiler]

gunluk_sayi = saatlik.resample("D").count()
gunluk_medyan = saatlik.resample("D").median()

# Yetersiz günleri silmiyoruz; takvimde tutup eksik işaretliyoruz.
gunluk_veri = gunluk_medyan.where(gunluk_sayi >= 18)

print("Saatlik kayıt sayısı:", len(veri))
print("Takvim günü sayısı:", len(gunluk_veri))

display(pd.DataFrame({
    "Geçerli gün": gunluk_veri.notna().sum(),
    "Yetersiz gün": gunluk_veri.isna().sum()
}))

display(gunluk_veri.head())

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Saatlik kayıt sayısı: 26305
Takvim günü sayısı: 1097


,Geçerli gün,Yetersiz gün
PM10,1059,38
PM2_5,1044,53
SO2,1026,71
NO2,1048,49


,PM10,PM2_5,SO2,NO2
tarih,,,,
2023-09-16,30.7,19.9,2.55,12.8
2023-09-17,23.0,17.5,3.60,7.0
2023-09-18,23.2,13.0,2.80,8.3
2023-09-19,NaN,NaN,NaN,NaN
2023-09-20,NaN,NaN,NaN,NaN


In [ ]:
# Her hedef gün için önceki 7 takvim gününün 4 kirleticisi:
# 7 gün × 4 kirletici = 28 girdi
X = pd.concat(
    [
        gunluk_veri.shift(gecikme).add_suffix(
            f"_onceki_{gecikme}_gun"
        )
        for gecikme in range(1, 8)
    ],
    axis=1
)

# Satırın tarihindeki gerçek değerler, tahmin hedeflerimiz.
y = gunluk_veri.copy()

# İlk 7 günde tam geçmiş pencere bulunmadığı için ayırıyoruz.
X = X.iloc[7:].copy()
y = y.iloc[7:].copy()

# Eksik satırları çıkarmadan önce takvim üzerinden %75 / %25 ayır.
ayrim = int(len(X) * 0.75)

X_egitim = X.iloc[:ayrim].copy()
X_test = X.iloc[ayrim:].copy()

y_egitim = y.iloc[:ayrim].copy()
y_test = y.iloc[ayrim:].copy()

print("Girdi sayısı:", X.shape[1])

print(
    "Eğitim hedef tarihleri:",
    X_egitim.index.min().date(),
    "–",
    X_egitim.index.max().date()
)

print(
    "Test hedef tarihleri:",
    X_test.index.min().date(),
    "–",
    X_test.index.max().date()
)

display(pd.DataFrame({
    "Eğitim": [
        len(X_egitim),
        X_egitim.notna().all(axis=1).sum(),
        y_egitim.notna().all(axis=1).sum()
    ],
    "Test": [
        len(X_test),
        X_test.notna().all(axis=1).sum(),
        y_test.notna().all(axis=1).sum()
    ]
}, index=[
    "Hedef takvim günü",
    "28 girdinin tamamı mevcut",
    "4 hedefin tamamı mevcut"
]))

display(X.head(3))

Girdi sayısı: 28
Eğitim hedef tarihleri: 2023-09-23 – 2025-12-17
Test hedef tarihleri: 2025-12-18 – 2026-09-16


,Eğitim,Test
Hedef takvim günü,817,273
28 girdinin tamamı mevcut,556,202
4 hedefin tamamı mevcut,704,253


,PM10_onceki_1_gun,PM2_5_onceki_1_gun,SO2_onceki_1_gun,NO2_onceki_1_gun,PM10_onceki_2_gun,PM2_5_onceki_2_gun,SO2_onceki_2_gun,NO2_onceki_2_gun,PM10_onceki_3_gun,PM2_5_onceki_3_gun,...,SO2_onceki_5_gun,NO2_onceki_5_gun,PM10_onceki_6_gun,PM2_5_onceki_6_gun,SO2_onceki_6_gun,NO2_onceki_6_gun,PM10_onceki_7_gun,PM2_5_onceki_7_gun,SO2_onceki_7_gun,NO2_onceki_7_gun
tarih,,,,,,,,,,,,,,,,,,,,,
2023-09-23,43.55,24.00,4.95,37.25,24.10,15.95,4.10,26.95,NaN,NaN,...,2.8,8.3,23.0,17.5,3.6,7.0,30.7,19.9,2.55,12.8
2023-09-24,73.70,32.10,6.40,54.10,43.55,24.00,4.95,37.25,24.10,15.95,...,NaN,NaN,23.2,13.0,2.8,8.3,23.0,17.5,3.60,7.0
2023-09-25,29.20,18.15,4.50,15.15,73.70,32.10,6.40,54.10,43.55,24.00,...,NaN,NaN,NaN,NaN,NaN,NaN,23.2,13.0,2.80,8.3


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

# Yalnızca eğitim dönemi içinde kronolojik doğrulama ayrımı
dogrulama_siniri = int(len(X_egitim) * 0.80)

X_ogrenme = X_egitim.iloc[:dogrulama_siniri]
X_dogrulama = X_egitim.iloc[dogrulama_siniri:]

y_ogrenme = y_egitim.iloc[:dogrulama_siniri]
y_dogrulama = y_egitim.iloc[dogrulama_siniri:]

sonuclar = []
ilk_modeller = {}

for k in kirleticiler:
    # Girdileri ve ilgili kirleticinin hedefi mevcut olan günler
    ogrenme_maskesi = (
        X_ogrenme.notna().all(axis=1)
        & y_ogrenme[k].notna()
    )

    dogrulama_maskesi = (
        X_dogrulama.notna().all(axis=1)
        & y_dogrulama[k].notna()
    )

    X_fit = X_ogrenme.loc[ogrenme_maskesi]
    y_fit = y_ogrenme.loc[ogrenme_maskesi, k]

    X_val = X_dogrulama.loc[dogrulama_maskesi]
    y_val = y_dogrulama.loc[dogrulama_maskesi, k]

    assert len(X_fit) > 0 and len(X_val) > 0, (
        f"{k}: yeterli eğitim/doğrulama örneği yok."
    )

    # Ölçekleme yalnızca öğrenme verisinden öğrenilir.
    # alpha=1.0 ilk deneme için sabit başlangıç ayarıdır.
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    )

    model.fit(X_fit, y_fit)
    model_tahmini = model.predict(X_val)

    # Aynı doğrulama günlerinde "yarın = bugün" tahmini
    baslangic_tahmini = X_val[f"{k}_onceki_1_gun"]

    baslangic_mae = mean_absolute_error(y_val, baslangic_tahmini)
    model_mae = mean_absolute_error(y_val, model_tahmini)

    sonuclar.append({
        "Kirletici": k,
        "Öğrenme günü": len(X_fit),
        "Doğrulama günü": len(X_val),
        "Başlangıç MAE": baslangic_mae,
        "Ridge MAE": model_mae,
        "İyileşme (%)": (
            100 * (baslangic_mae - model_mae) / baslangic_mae
            if baslangic_mae > 0 else np.nan
        ),
        "Negatif tahmin sayısı": int((model_tahmini < 0).sum())
    })

    ilk_modeller[k] = model

dogrulama_sonuclari = pd.DataFrame(sonuclar)

print(
    "Doğrulama hedef tarihleri:",
    X_dogrulama.index.min().date(),
    "–",
    X_dogrulama.index.max().date()
)

display(dogrulama_sonuclari.round(3))

Doğrulama hedef tarihleri: 2025-07-07 – 2025-12-17


,Kirletici,Öğrenme günü,Doğrulama günü,Başlangıç MAE,Ridge MAE,İyileşme (%),Negatif tahmin sayısı
0,PM10,503,49,7.569,7.723,-2.030,0
1,PM2_5,500,49,2.633,3.399,-29.127,0
2,SO2,500,48,1.194,1.519,-27.260,0
3,NO2,502,49,9.579,11.423,-19.259,0


In [ ]:
alpha_degerleri = [0.1, 1.0, 10.0, 100.0, 1000.0]
ayar_sonuclari = []

for k in kirleticiler:
    ogrenme_maskesi = (
        X_ogrenme.notna().all(axis=1)
        & y_ogrenme[k].notna()
    )
    dogrulama_maskesi = (
        X_dogrulama.notna().all(axis=1)
        & y_dogrulama[k].notna()
    )

    X_fit = X_ogrenme.loc[ogrenme_maskesi]
    y_fit = y_ogrenme.loc[ogrenme_maskesi, k]

    X_val = X_dogrulama.loc[dogrulama_maskesi]
    y_val = y_dogrulama.loc[dogrulama_maskesi, k]

    baslangic_mae = mean_absolute_error(
        y_val, X_val[f"{k}_onceki_1_gun"]
    )

    for alpha in alpha_degerleri:
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=alpha)
        )
        model.fit(X_fit, y_fit)
        tahmin = model.predict(X_val)

        ayar_sonuclari.append({
            "Kirletici": k,
            "Alpha": alpha,
            "Ridge MAE": mean_absolute_error(y_val, tahmin),
            "Başlangıç MAE": baslangic_mae,
            "Doğrulama günü": len(y_val)
        })

ayar_tablosu = pd.DataFrame(ayar_sonuclari)

display(
    ayar_tablosu.pivot(
        index="Kirletici",
        columns="Alpha",
        values="Ridge MAE"
    ).round(3)
)

en_iyi_ayarlar = (
    ayar_tablosu.loc[
        ayar_tablosu.groupby("Kirletici")["Ridge MAE"].idxmin()
    ]
    .set_index("Kirletici")
    .reindex(kirleticiler)
)

print("Her kirletici için en düşük doğrulama MAE:")
display(en_iyi_ayarlar.round(3))

Alpha,0.1,1.0,10.0,100.0,1000.0
Kirletici,,,,,
NO2,11.424,11.423,11.440,11.754,12.987
PM10,7.732,7.723,7.667,7.858,9.068
PM2_5,3.396,3.399,3.444,3.882,5.325
SO2,1.514,1.519,1.568,1.933,3.074


Her kirletici için en düşük doğrulama MAE:


,Alpha,Ridge MAE,Başlangıç MAE,Doğrulama günü
Kirletici,,,,
PM10,10.0,7.667,7.569,49
PM2_5,0.1,3.396,2.633,49
SO2,0.1,1.514,1.194,48
NO2,1.0,11.423,9.579,49


## Tahminleme yaklaşımı ve ilk deneme notu

- **Hedef:** Gün tamamlandığında bilinen ölçümlerle, ertesi günün PM10, PM2.5, SO2 ve NO2 günlük medyanlarını tahmin etmek.
- **Girdiler:** Önceki yedi takvim gününün dört kirletici değeri; toplam **28 girdi**. Yedi gün, yakın geçmişi ve bir haftalık bilgiyi kapsayan ilk deneme tercihidir; en iyi süre olduğu henüz gösterilmemiştir.
- **Veri hazırlama:** EDA’daki kurallar korundu. Günlük medyan için en az 18 saatlik ölçüm arandı; şüpheli 985 değerleri analiz kopyasında dışlandı. İlk model denemesinde eksikler doldurulmadı; girdileri ve ilgili hedefi mevcut günler kullanıldı.
- **%75 eğitim / %25 test:** Verinin çoğunu öğrenmeye ayırırken sonraki dönemde değerlendirme yapabilmek için seçildi. Evrensel bir kural değildir. Gelecekten geçmişe bilgi taşınmaması için tarihler sıralı tutuldu, rastgele karıştırılmadı.
- **Doğrulama:** İlk %75’in son %20’si model ayarlarını karşılaştırmak için ayrıldı. Böylece toplam dönemin yaklaşık %60’ı öğrenme, %15’i doğrulama, %25’i test oldu. Testteki tahmin hataları henüz hesaplanmadı.
- **Neden MAE?** Tahminlerin gerçek değerlerden ortalama kaç **µg/m³** saptığını anlaşılır biçimde gösterir. Her kirletici için ayrı hesaplanır; düşük değer daha iyidir.
- **Neden Ridge?** Basit ve açıklanabilir bir ilk modeldir. Birbiriyle ilişkili geçmiş ölçümler kullanılırken katsayıların aşırı büyümesini sınırlar. İlk adaydır; nihai model olarak seçilmiş değildir.
- **Başlangıç karşılaştırması:** Her kirletici için “yarının değeri bugünküyle aynı” tahmini kullanıldı. Model ve başlangıç tahmini aynı günlerde değerlendirildi.

### İlk sonuçlar

| Kirletici | Seçilen alpha | Ridge MAE | Başlangıç MAE |
|---|---:|---:|---:|
| PM10 | 10 | 7,667 | **7,569** |
| PM2.5 | 0,1 | 3,396 | **2,633** |
| SO2 | 0,1 | 1,514 | **1,194** |
| NO2 | 1 | 11,423 | **9,579** |

Denenen ayarlar içinde en düşük doğrulama MAE’si seçilmesine rağmen Ridge, hiçbir kirleticide başlangıç tahminini geçememiştir. Büyük alpha değerleri bu denemede sonuçları kötüleştirmiştir.

Doğrulamadaki 164 takvim gününden yalnızca 48–49’u değerlendirmeye girmiştir; sonuçlar girdileri tam olan bu günleri temsil eder. Bu nedenle Ridge’in genel olarak işe yaramadığı sonucuna varılamaz. Sonraki adım, test dönemini model seçimine katmadan yaklaşımı geliştirmektir.

İlk deneyde, önceki yedi günün dört kirletici değerini kullanan Ridge regresyonu denenmiştir. Beş farklı düzenlileştirme ayarı karşılaştırılmış; değerlendirilebilen 48–49 doğrulama gününde hiçbir kirletici için “yarın = bugün” başlangıç tahmininden daha düşük MAE elde edilememiştir. Bu nedenle Ridge’in bu kurulumda ek tahmin başarısı sağladığı gösterilememiştir. Eksik geçmiş verilerin kapsam üzerindeki etkisi ve alternatif yaklaşımlar sonraki aşamada incelenecektir.